# Visualize the DQN agent

Walks through a single episode of the trained agent and visualises:

1. The encoded observation (per-channel heatmap).
2. The action mask.
3. The top-5 Q-values for the current state.
4. A training-curve summary.

Run from the repo root; the notebook adds the root to `sys.path` so `import rl` works regardless of CWD.

In [ ]:
import os, sys
from pathlib import Path
root = Path.cwd()
while not (root / 'rl').exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
print('repo root:', root)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from rl.agents import DQNAgent
from rl.agents.dqn_agent import DQNConfig
from rl.encoding import NUM_ACTIONS, NUM_CELLS, OBS_CHANNELS, ActionEncoder
from rl.env import CheckersEnv, EnvConfig

CKPT = root / 'rl/checkpoints/dqn_best.pt'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = DQNConfig(obs_dim=OBS_CHANNELS*NUM_CELLS, num_actions=NUM_ACTIONS, device=device)
agent = DQNAgent(cfg)
if CKPT.exists():
    agent.load(str(CKPT))
    print(f'Loaded {CKPT}')
else:
    print('No checkpoint — running with random weights.')

env = CheckersEnv(EnvConfig(mode='solo_race', my_colour='red', max_steps=600, seed=0))
obs, info = env.reset(seed=0)
mask = info['legal_mask']
print('obs shape', obs.shape, 'legal count', int(mask.sum()))

## Observation channels

16 channels of length 121. Each scatter plot uses the board's pixel coordinates so you can see spatial layout. Non-zero cells are highlighted.

In [ ]:
xs = np.array([c.x for c in env.board.cells])
ys = np.array([c.y for c in env.board.cells])

titles = [
    'my_pins', 'opponent_pins', 'my_goal', 'my_start', 'empty', 'move_progress',
    *[f'pin_{i}' for i in range(10)],
]
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ch, ax in enumerate(axes.flat):
    sc = ax.scatter(xs, -ys, c=obs[ch], s=60, cmap='viridis', vmin=0, vmax=1)
    ax.set_title(titles[ch])
    ax.set_aspect('equal'); ax.axis('off')
fig.tight_layout(); plt.show()

## Action mask and Q-values

We show the current legal actions in the `(pin_id, to_cell)` grid, and highlight the five actions with the highest Q-value.

In [ ]:
with torch.no_grad():
    q = agent.online(torch.from_numpy(obs.reshape(1, -1)).to(device)).cpu().numpy().reshape(-1)

q_masked = np.where(mask, q, -np.inf)
topk = np.argsort(q_masked)[-5:][::-1]
print('Top-5 Q-values (pin_id, to_index, q):')
for flat in topk:
    pid, to = ActionEncoder.from_flat(int(flat))
    print(f'  pin {pid} -> cell {to}   Q={q[flat]:.3f}')

mask_grid = mask.reshape(10, NUM_CELLS)
q_grid = np.where(mask_grid, q.reshape(10, NUM_CELLS), np.nan)
fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(q_grid, aspect='auto', cmap='coolwarm')
ax.set_ylabel('pin_id'); ax.set_xlabel('to cell index')
plt.colorbar(im, ax=ax, label='Q (illegal masked)')
plt.show()

## Play through one episode and render

In [ ]:
obs, info = env.reset(seed=1)
mask = info['legal_mask']
print('== initial ==')
print(env.render_ascii())

for step in range(600):
    action = agent.act(obs, mask, training=False)
    if action < 0: break
    res = env.step(action)
    obs = res.obs
    mask = env.action_mask()
    if res.terminated or res.truncated:
        print(f'Episode ended: outcome={res.info.get("outcome")}  '
              f'agent_moves={env.agent_move_count}  pins_home={res.info.get("pins_home")}')
        break
print('== final ==')
print(env.render_ascii())

## Training-curve summary

Reads `rl/checkpoints/metrics/episodes.csv` and plots return, pins_home, moves, and rolling win rate.

In [ ]:
import csv
from collections import deque
csv_path = root / 'rl/checkpoints/metrics/episodes.csv'
rows = []
if csv_path.exists():
    with open(csv_path) as f:
        for r in csv.DictReader(f):
            rows.append(r)
print('episodes logged:', len(rows))

if rows:
    ep = [int(r['episode']) for r in rows]
    ret = [float(r['return_sum']) for r in rows]
    pin = [int(r['pins_home']) for r in rows]
    mov = [int(r['agent_moves']) for r in rows]
    win = [1 if r['outcome'] == 'win' else 0 for r in rows]

    roll, buf = [], deque(maxlen=100)
    for w in win:
        buf.append(w); roll.append(sum(buf)/len(buf))

    fig, axes = plt.subplots(2, 2, figsize=(11, 6))
    axes[0,0].plot(ep, ret); axes[0,0].set_title('Return per episode')
    axes[0,1].plot(ep, pin); axes[0,1].set_title('Pins in goal')
    axes[1,0].plot(ep, mov); axes[1,0].set_title('Agent moves')
    axes[1,1].plot(ep, roll); axes[1,1].set_title('Rolling win-rate (w=100)')
    for a in axes.flat: a.grid(True, alpha=0.3)
    fig.tight_layout(); plt.show()